Tutorial 5: Expectation Propagation
===================================

In the previous tutorials, we fitted graphical models to dataset comprising many noisy 1D Gaussians. These had a shared
and global value of their `centre`, or assumed their centres were hierarchically drawn from a parent Gaussian
distribution. This provides the basis of composing and fitting complex graphical models to large datasets.

We concluded by discussing that there is a ceiling scaling these graphical models up to extremely large datasets. One
would soon find that the parameter space is too complex to sample, and computational limits would ultimately cap how
many datasets one could feasibly fit.

This tutorial introduces expectation propagation (EP), the solution to this problem, which inspects a factor graph
and partitions the model-fit into many simpler fits of sub-components of the graph to individual datasets. This
overcomes the challenge of model complexity, and mitigates computational restrictions that may occur if one tries to
fit every dataset simultaneously.

This tutorial fits a global model with a shared parameter and does not use a hierarchical model. The optional tutorial
`tutorial_optional_hierarchical_ep` shows an example fit of a hierarchical model with EP.

__Contents__

This tutorial is split into the following sections:

- **Example Source Code (`af.ex`)**: The example objects used in this tutorial.
- **Dataset**: Load the noisy 1D Gaussian datasets with a shared centre.
- **Analysis**: Create Analysis objects for each dataset.
- **Model**: Set up the model with a shared centre prior across all datasets.
- **Analysis Factors**: Create Analysis Factors with individual searches for each dataset.
- **A Width Relation**: Connect the inferred widths with an explicitly assumed linear relation.
- **Factor Graph**: Compose the factor graph for the EP framework.
- **Expectation Propagation**: Explain the EP message passing algorithm.
- **Cyclic Fitting**: Describe the iterative EP convergence process.
- **Result**: Access the result of the EP fit.
- **Output**: Describe the output directory structure and files generated by the EP fit.
- **Results**: Use the MeanField object to infer parameter estimates and errors.

__Google Colab Setup__

This cell sets up the environment when the notebook is run on Google Colab: it installs the
required PyAuto packages, clones the workspace (configuration files and example datasets) and
points the configuration at it. If you are running the notebook elsewhere (e.g. locally via
your own installation) it does nothing, and you can run it safely.

Colab tip: model-fits run much faster on a GPU — enable one via "Runtime" -> "Change runtime
type" -> "Hardware accelerator" before running the notebook.

In [ ]:
try:
    import google.colab
except ImportError:
    from autofit import setup_colab as _setup_colab
else:
    import importlib
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "autonerves", "--no-deps"]
    )
    _setup_colab = importlib.import_module("autonerves.setup_colab")

_setup_colab.setup("howtofit")

In [ ]:

from autofit import setup_notebook; setup_notebook()

import numpy as np
from os import path

import autofit as af

__Example Source Code (`af.ex`)__

The **PyAutoFit** source code has the following example objects (accessed via `af.ex`) used in this tutorial:

 - `Analysis`: an analysis object which fits noisy 1D datasets, including `log_likelihood_function` and 
 `visualize` functions.

 - `Gaussian`: a model component representing a 1D Gaussian profile.

 - `plot_profile_1d`: a function for plotting 1D profile datasets including their noise.

These are functionally identical to the `Analysis`, `Gaussian` and `plot_profile_1d` objects and functions you have seen 
and used elsewhere throughout the workspace.

__Dataset__

For each dataset we now set up the correct path and load it. 

We first fit the 1D Gaussians which all share the same centre, thus not requiring a hierarchical model. 

An example for fitting the hierarchical model with EP is given at the end of this tutorial.

In [ ]:
total_datasets = 3

__Dataset Auto-Simulation__

If the dataset does not already exist on your system, it will be created by running the corresponding
simulator script. This ensures that all example scripts can be run without manually simulating data first.

In [ ]:
if not path.exists(
    path.join("dataset", "example_1d", "gaussian_x1__low_snr", "dataset_0")
):
    import subprocess
    import sys

    subprocess.run(
        [sys.executable, "scripts/simulators/simulators.py"],
        check=True,
    )

dataset_name_list = []
data_list = []
noise_map_list = []

for dataset_index in range(total_datasets):
    dataset_name = f"dataset_{dataset_index}"

    dataset_path = path.join(
        "dataset", "example_1d", "gaussian_x1__low_snr", dataset_name
    )

    data = af.util.numpy_array_from_json(file_path=path.join(dataset_path, "data.json"))
    noise_map = af.util.numpy_array_from_json(
        file_path=path.join(dataset_path, "noise_map.json")
    )

    dataset_name_list.append(dataset_name)
    data_list.append(data)
    noise_map_list.append(noise_map)

By plotting the Gaussians we can remind ourselves that determining their centres by eye is difficult.

In [ ]:
for dataset_name, data in zip(dataset_name_list, data_list):
    af.ex.plot_profile_1d(
        xvalues=np.arange(data.shape[0]),
        profile_1d=data,
        title=dataset_name,
        ylabel="Data Values",
        color="k",
    )

__Analysis__

For each dataset we now create a corresponding `Analysis` class, like in the previous tutorial.

In [ ]:
analysis_list = []

for data, noise_map in zip(data_list, noise_map_list):
    analysis = af.ex.Analysis(data=data, noise_map=noise_map)

    analysis_list.append(analysis)

__Model__

We now compose the graphical model that we fit, using the `Model` and `Collection` objects you are now familiar with.

We will assume all Gaussians share the same centre, therefore we set up a shared prior for `centre`.

In [ ]:
centre_shared_prior = af.GaussianPrior(mean=50.0, sigma=30.0)

model_list = []

for model_index in range(len(data_list)):
    gaussian = af.Model(af.ex.Gaussian)

    gaussian.centre = centre_shared_prior
    gaussian.normalization = af.TruncatedGaussianPrior(
        mean=3.0, sigma=5.0, lower_limit=0.0
    )
    gaussian.sigma = af.TruncatedGaussianPrior(mean=10.0, sigma=10.0, lower_limit=0.0)

    model = af.Collection(gaussian=gaussian)

    model_list.append(model)

__Analysis Factors__

Now we have our `Analysis` classes and graphical model, we can compose our `AnalysisFactor`'s.

However, unlike the previous tutorials, each `AnalysisFactor` is now assigned its own `search`. This is because the EP 
framework performs a model-fit to each node on the factor graph (e.g. each `AnalysisFactor`). Therefore, each node 
requires its own non-linear search, and in this tutorial we use `dynesty`. For complex graphs consisting of many 
nodes, one could easily use different searches for different nodes on the factor graph.

Each `AnalysisFactor` is also given a `name`, corresponding to the name of the dataset it fits. These names are used
to name the folders containing the results in the output directory.

In [ ]:
paths = af.DirectoryPaths(
    name=path.join("chapter_advanced", "tutorial_5_expectation_propagation")
)

search = af.DynestyStatic(paths=paths, nlive=100, sample="rwalk")

analysis_factor_list = []

dataset_index = 0

for model, analysis in zip(model_list, analysis_list):
    dataset_name = f"dataset_{dataset_index}"
    dataset_index += 1

    analysis_factor = af.AnalysisFactor(
        prior_model=model, analysis=analysis, optimiser=search, name=dataset_name
    )

    analysis_factor_list.append(analysis_factor)


__A Width Relation__

To show how another factor can connect to the same datasets, we also model their full widths at half maximum
(FWHMs). For this teaching example, assume the widths scatter around a line `m * x + c`, where `x` is the
dataset index (0, 1 or 2). The scatter is fixed at 5 pixels. This is an additional modelling assumption, not
a second set of measured widths: the widths remain parameters inferred from the noisy profiles above.

The regression factor evaluates a normalized Gaussian density for each width about that line. It therefore
passes information about the widths between datasets, while the shared centre connects the profile factors
directly. The simulated widths are similar, so a slope near zero is plausible; we do not fix it to zero.

In [ ]:


class LinearRegressionAnalysis(af.Analysis):
    def __init__(self, xvalues, scatter):
        self.xvalues = xvalues
        self.scatter = scatter

    def log_likelihood_function(self, instance, xp=np):
        widths = xp.asarray(list(instance.fwhm))
        prediction = instance.m * xp.asarray(self.xvalues) + instance.c
        residuals = (widths - prediction) / self.scatter
        return -0.5 * xp.sum(residuals**2 + xp.log(2.0 * xp.pi * self.scatter**2))


fwhm_list = [2 * np.sqrt(2 * np.log(2)) * model.gaussian.sigma for model in model_list]

linear_regression_factor = af.AnalysisFactor(
    prior_model=af.Collection(
        m=af.GaussianPrior(mean=0.0, sigma=5.0),
        c=af.GaussianPrior(mean=2 * np.sqrt(2 * np.log(2)) * 10.0, sigma=10.0),
        fwhm=af.Collection(*fwhm_list),
    ),
    analysis=LinearRegressionAnalysis(xvalues=np.arange(total_datasets), scatter=5.0),
    optimiser=search,
    name="linear_regression",
)

__Factor Graph__

We combine our `AnalysisFactors` into one, to compose the factor graph.

In [ ]:
factor_graph = af.FactorGraphModel(
    *analysis_factor_list,
    linear_regression_factor,
)

The factor graph model `info` attribute shows the model which we fit via expectaton propagation (note that we do
not use `global_prior_model` below when performing the fit).

In [ ]:
print(factor_graph.global_prior_model.info)

The same model can also be visualized as a figure, making its structure easier to understand at a glance.

The figure shows how the model is organized: which parameters belong to each component, and whether they are free,
fixed, shared, linked by an expression, solved during the fit, or not configured. `model.info` provides the
corresponding numerical details, including the prior assigned to each free parameter and the value of each fixed
parameter.

EP never fits this whole model in one go. It visits one factor at a time, passing messages along the links between
the factors, and those messages are what the rest of this tutorial follows.

In [ ]:
af.ModelPlotter(factor_graph.global_prior_model).figure()

__Expectation Propagation__

In the previous tutorials, we used the `global_prior_model` of the `factor_graph` to fit the global model. In this 
tutorial, we instead fit the `factor_graph` using the EP framework, which fits the graphical model composed in this 
tutorial as follows:

1) Go to the first node on the factor graph (e.g. `analysis_factor_list[0]`) and fit its model to its dataset. This is 
simply a fit of the `Gaussian` model to the first 1D Gaussian dataset, the model-fit we are used to performing by now.

2) Once the model-fit is complete, inspect the model for parameters that are shared with other nodes on the factor
graph. In this example, the `centre` of the `Gaussian` fitted to the first dataset is global, and therefore connects
to the other nodes on the factor graph (the `AnalysisFactor`'s) of the second and third `Gaussian` datasets.

3) The EP framework now creates a 'message' that is to be passed to the connecting nodes on the factor graph. This
message informs them of the results of the model-fit, so they can update their priors on the `Gaussian`'s centre 
accordingly and, more importantly, update their posterior inference and therefore estimate of the global centre.

For example, the model fitted to the first Gaussian dataset includes the global centre. Therefore, after the model is 
fitted, the EP framework creates a 'message' informing the factor graph about its inference on that Gaussians's centre,
thereby updating our overall inference on this shared parameter. This is termed 'message passing'.

__Cyclic Fitting__

After every `AnalysisFactor` has been fitted (the three datasets and the width relation in this example), we have a
new estimate of the shared parameter `centre`. This updates our priors on the shared parameter `centre`, which needs 
to be reflected in each model-fit we perform on each `AnalysisFactor`. 

The EP framework therefore performs a second iteration of model-fits. It again cycles through each `AnalysisFactor` 
and refits the model, using updated priors on shared parameters like the `centre`. At the end of each fit, we again 
create messages that update our knowledge about other parameters on the graph.

This process is repeated multiple times, until a convergence criteria is met whereby continued cycles are expected to
produce the same estimate of the shared parameter `centre`. 

When we fit the factor graph a `name` is passed, which determines the folder all results of the factor graph are
stored in.

We also pass `visualise_interval=1`. The EP fit draws the factor graph with the state of the run painted onto it and
writes it to the output folder, and this input sets how many sweeps pass between refreshes of that figure. Its
default is 100, which over the 5 sweeps below would draw it exactly once, so we set it to 1 and get a fresh picture
of the graph after every sweep.

In [ ]:
laplace = af.LaplaceOptimiser()

factor_graph_result = factor_graph.optimise(
    optimiser=laplace,
    paths=paths,
    ep_history=af.EPHistory(kl_tol=0.05),
    max_steps=5,
    visualise_interval=1,
)

Five sweeps demonstrate message passing, but do not guarantee convergence. Before interpreting the result,
check that every data or regression factor contributed at least one update. A run that exits normally can
still contain a stale factor; the guard below makes that failure explicit, including in automated smoke tests.
An update can also reject an individual marginal projection, which the state figure below marks separately.

In [ ]:
factor_updates = {
    factor.name: len(history.updates)
    for factor, history in factor_graph_result.ep_history.items()
}
stale_factors = [
    name
    for name in [*dataset_name_list, "linear_regression"]
    if factor_updates.get(name, 0) == 0
]
if stale_factors:
    raise RuntimeError(f"EP factors never updated: {', '.join(stale_factors)}")

__Result__

An `info` attribute for the result of a factor graph fitted via EP does not exist yet, its on the to do list!

The result can be seen in the `graph.result` file output to hard-disk.

In [ ]:
### print(factor_graph_result.info)##

__Seeing the EP run__

Until that `info` exists, the figure is how we read an EP fit back. `af.EPPlotter` draws the same factor graph as
before, but now with the state of the run painted onto it.

The graph it draws is not the model figure we drew earlier, it is the factor graph itself, and its legend says how
to read it: `box = factor`, `pill = variable`, `dashed frame = repeated structure`. Every `AnalysisFactor` is a box, every
parameter it touches is a pill, and an edge between them means "this factor has something to say about this
variable". That is the picture the message passing above lives on: a message leaves a box, arrives at a pill, and is
read by every other box attached to it. Follow the shared `centre` pill to see the connection between the three
dataset factors. The width relation connects their `sigma` parameters through the FWHM expressions. Repeated
structure may be grouped into a plate; individual factors are expanded when their run states differ.

Onto that structure the `ep_history` paints what the sweeps actually did. Every box carries a status:

 - `working`: it is updating, nothing was rejected, and it has not yet converged.
 - `converged`, drawn green: the `EPHistory` convergence test, our `kl_tol=0.05` above, passes for this factor.
 - `stale`, drawn grey: the sweeps reached it but it never landed a single update, so the posterior reported for its
 variables is still the message it started with.
 - `reverting`, drawn with red dashed edges: its latest projection was rejected for at least one variable and that
 variable's message was put back, so the box updated but that marginal did not move.

Each box is also badged with how many updates it landed out of how many sweeps, an `age` counting the sweeps since
it last moved, and the reason code of its last rejection, such as `BAD_PROJECTION`. The footer totals the whole
graph: how many factors, variables and plates it has, and how many are stale, reverting, converged and working after
the sweeps that were run.

Look at the grey boxes first. A `stale` factor means a dataset contributed nothing at all to the shared inference,
which is a very different failure from a fit that simply has not converged yet, and it is easy to miss in the numbers
printed below because EP can report convergence in exactly that state: if no factor updates, the step between sweeps
is zero.

Note which factor graph we hand the plotter. `factor_graph_result.factor_graph` is the graph the optimiser actually
swept, and the `ep_history` is keyed by its factor objects. Do not pass `factor_graph.graph` instead: that property
builds a fresh graph and renames its factors on every access, so the history would match nothing and every node would
be drawn as `absent`.

Passing `kind="model"` instead draws the structure alone, with no run painted on it, which is the view to use before
a fit starts.

In [ ]:
af.EPPlotter(
    factor_graph_result.factor_graph, ep_history=factor_graph_result.ep_history
).figure(kind="state")

__Output__

The results of the factor graph, using the EP framework and message passing, are contained in the folder 
`output/chapter_advanced/tutorial_5_expectation_propagation`. 

The following folders and files are worth of note:

 - `graph.info`: this provides an overall summary of the graphical model that is fitted, including every parameter, 
 how parameters are shared across `AnalysisFactor`'s and the priors associated to each individual parameter.
 
 - The 3 folders titled `gaussian_x1_#__low_snr` correspond to the three `AnalysisFactor`'s and therefore signify 
 repeated non-linear searches that are performed to fit each dataset.
 
 - Inside each of these folders are `optimization_#` folders, corresponding to each model-fit performed over cycles of
 the EP fit. A careful inspection of the `model.info` files inside each folder reveals how the priors are updated
 over each cycle, whereas the `model.results` file should indicate the improved estimate of model parameters over each
 cycle.

 - `graph_model.png`: the factor graph drawn as structure alone, written once at the start of the fit beside
 `graph.info`.

 - `graph_state.png`: the same graph with the state of the run painted onto it, rewritten every `visualise_interval`
 sweeps and therefore, with the `visualise_interval=1` we passed above, after every sweep. Flicking through it as the
 fit runs is the quickest way to see which factors are moving and which have gone stale.

Both `.png` files are only written when `model_figure` is set to `true` in `config/output.yaml`; it ships as
`false`, so switch it on before the fit if you want them.

__Results__

The `MeanField` object represent the posterior of the entire factor graph and is used to infer estimates of the 
values and error of each parameter in the graph.

In [ ]:
mean_field = factor_graph_result.updated_ep_mean_field.mean_field
print(mean_field)
print()

The object has a `variables` property which lists every variable in the factor graph, which is essentially all of the 
free parameters on the graph.

This includes the parameters specific to each data (E.g. each node on the graph) as well as the shared centre.

In [ ]:
print(mean_field.variables)
print()

# """
# The variables above use the priors on each parameter as their key.
#
# Therefore to estimate mean-field quantities of the shared centre, we can simply use the `centre_shared_prior` defined
# above.
#
# Each parameter estimate is given by the mean of its value in the `MeanField`. Below, we use the `centred_shared_prior`
# as a key to the `MeanField.mean` dictionary to print the estimated value of the shared centre.
# """
# print(f"Centre Mean Parameter Estimate = {mean_field.mean[centre_shared_prior]}")
# print()
#
# """
# If we want the parameter estimate of another parameter in the model, we can use the `model_list` that we composed
# above to pass a parameter prior to the mean field dictionary.
# """
# print(
#     f"Normalization Gaussian Dataset 0 Mean = {mean_field.mean[model_list[0].gaussian.normalization]}"
# )
#
# """
# The mean-field mean dictionary contains the estimate value of every parameter.
# """
# print(f"All Parameter Estimates = {mean_field.mean}")
# print()
#
# """
# The mean-field also contains a `variance` dictionary, which has the same keys as the `mean` dictionary above.
#
# This is the easier way to estimate the error on every parameter, for example that of the shared centre.
# """
# print(f"Centre Variance = {mean_field.variance[centre_shared_prior]}")
# print()
#
# """
# The standard deviation (or error at one sigma confidence interval) is given by the square root of the variance.
# """
# print(f"Centre 1 Sigma = {np.sqrt(mean_field.variance[centre_shared_prior])}")
# print()
#
# """
# The mean field object also contains a dictionary of the s.d./variance**0.5.
# """
# print(f"Centre SD/sqrt(variance) = {mean_field.scale[centre_shared_prior]}")
# print()

__Wrap Up__

This tutorial introduced Expectation Propagation (EP), which fits a graphical model by passing messages between its
factors rather than sampling the full joint parameter space at once. This makes inference scalable to graphs with many
datasets and shared parameters.

The commented-out code above shows optional further inspection of the `MeanField` result (per-parameter means,
variances and standard deviations) which you can uncomment to explore.